In [1]:
!nvcc --version
!nvidia-smi
import torch; print(torch.__version__, torch.version.cuda)

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Sun Jun 28 04:03:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0       

In [24]:
!pip install -q "bitsandbytes>=0.46.1"
!pip install -q "vllm==0.11.0" "transformers>=4.51,<5" "sentence-transformers>=3,<4" \
  --extra-index-url https://download.pytorch.org/whl/cu128
!pip install -U accelerate faiss-cpu
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 932.0/932.0 kB 56.9 MB/s eta 0:00:00


In [2]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True NVIDIA A100-SXM4-40GB


In [4]:
%%writefile config.py
from dataclasses import dataclass

@dataclass
class Config:
    embed_model: str = "BAAI/bge-small-en-v1.5"
    gen_model: str = "Qwen/Qwen2.5-1.5B-Instruct"
    chunk_size: int = 512
    chunk_overlap: int = 64
    top_k: int = 4
    max_new_tokens: int = 256
    index_dir: str = "store"

CFG = Config()

Writing config.py


In [5]:
%%writefile embedder.py
from sentence_transformers import SentenceTransformer
import numpy as np
from config import CFG

QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

class Embedder:
    def __init__(self, model_name: str = CFG.embed_model, device: str = "cuda"):
        self.model = SentenceTransformer(model_name, device=device)
        self.dim = self.model.get_sentence_embedding_dimension()

    def embed_documents(self, texts: list[str]) -> np.ndarray:
        return self.model.encode(
            texts, normalize_embeddings=True, convert_to_numpy=True,
            batch_size=64, show_progress_bar=False
        ).astype("float32")

    def embed_query(self, text: str) -> np.ndarray:
        return self.model.encode(
            QUERY_PREFIX + text, normalize_embeddings=True, convert_to_numpy=True,
        ).astype("float32")


Writing embedder.py


In [6]:
%%writefile generator.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from config import CFG

class Generator:
    def __init__(self, model_name: str = CFG.gen_model, device: str = "cuda"):
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.float16
        ).to(device)
        self.model.eval()
        self.device = device

    def build_prompt(self, question: str, contexts: list[str]) -> str:
        context_block = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(contexts))
        system = (
            "You are a helpful assistant. Answer the question using ONLY the context "
            "below. If the answer isn't in the context, say you don't know. "
            "Cite sources like [1], [2] where relevant."
        )
        messages = [
            {"role": "system", "content": system},
            {"role": "user",
             "content": f"Context:\n{context_block}\n\nQuestion: {question}"},
        ]

        return self.tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    @torch.inference_mode()
    def generate(self, prompt: str) -> str:
        inputs = self.tok(prompt, return_tensors="pt").to(self.device)
        out = self.model.generate(
            **inputs, max_new_tokens=CFG.max_new_tokens,
            do_sample=False, pad_token_id=self.tok.eos_token_id
        )
        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        return self.tok.decode(new_tokens, skip_special_tokens=True)


Writing generator.py


In [7]:
%%writefile index.py

import faiss, numpy as np, json, os
from config import CFG

class VectorIndex:
    def __init__(self, dim: int):
        self.index = faiss.IndexFlatIP(dim)
        self.chunks: list[str] = []

    def add(self, vectors: np.ndarray, chunks: list[str]):
        self.index.add(vectors)
        self.chunks.extend(chunks)

    def search(self, query_vec: np.ndarray, k: int):
        scores, idx = self.index.search(query_vec.reshape(1, -1), k)
        return [(self.chunks[i], float(s))
                for s, i in zip(scores[0], idx[0]) if i != -1]

    def save(self, path: str = CFG.index_dir):
        os.makedirs(path, exist_ok=True)
        faiss.write_index(self.index, os.path.join(path, "faiss.index"))
        with open(os.path.join(path, "chunks.json"), "w") as f:
            json.dump(self.chunks, f)

    @classmethod
    def load(cls, dim: int, path: str = CFG.index_dir):
        obj = cls(dim)
        obj.index = faiss.read_index(os.path.join(path, "faiss.index"))
        with open(os.path.join(path, "chunks.json")) as f:
            obj.chunks = json.load(f)
        return obj


Writing index.py


In [1]:
%%writefile ingest.py

import os, glob, argparse
from embedder import Embedder
from index import VectorIndex
from config import CFG

def chunk_text(text: str, size: int, overlap: int) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start+size])
        start += size - overlap
    return chunks

def main(docs_dir: str):
    embedder = Embedder()
    index = VectorIndex(embedder.dim)
    paths = glob.glob(os.path.join(docs_dir, "**", "*.*"), recursive=True)
    all_chunks = []
    for p in paths:
        if not p.endswith((".txt", ".md")):
            continue
        with open(p, encoding="utf-8", errors="ignore") as f:
            all_chunks.extend(chunk_text(f.read(), CFG.chunk_size, CFG.chunk_overlap))
    print(f"{len(all_chunks)} chunks from {len(paths)} files")
    index.add(embedder.embed_documents(all_chunks), all_chunks)
    index.save()
    print(f"Index saved to {CFG.index_dir}/ ({index.index.ntotal} vectors)")

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--docs", default="docs")
    main(ap.parse_args().docs)



Overwriting ingest.py


In [9]:
%%writefile rag.py

from embedder import Embedder
from index import VectorIndex
from generator import Generator
from config import CFG

class RAG:
    def __init__(self):
        self.embedder = Embedder()
        self.index = VectorIndex.load(self.embedder.dim)
        self.generator = Generator()

    def answer(self, question: str, k: int = CFG.top_k) -> str:
        qvec = self.embedder.embed_query(question)
        hits = self.index.search(qvec, k)
        contexts = [text for text, _ in hits]
        prompt = self.generator.build_prompt(question, contexts)
        return self.generator.generate(prompt)


Overwriting rag.py


In [10]:
%%writefile ask.py
import sys
from rag import RAG

if __name__ == "__main__":
    rag = RAG()
    q = " ".join(sys.argv[1:]) or input("Question: ")
    print("\n" + rag.answer(q))

Overwriting ask.py


In [11]:
!python ingest.py --docs docs

2026-06-28 04:18:56.012304: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-28 04:18:56.082145: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
58 chunks from 5 files
Index saved to store/ (58 vectors)


In [12]:
!python ask.py "Why is the decode phase memory-bandwidth-bound?"

2026-06-28 04:19:11.656148: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-28 04:19:11.726113: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

The decode phase is memory-bandwidth-bound due to the following reasons:

1. **Memory Bandwidth**: During decode, the model reads all model weights from high-bandwidth mem

In [13]:
%%writefile bench.py
import time, statistics, torch
from embedder import Embedder
from index import VectorIndex
from generator import Generator
from config import CFG

QUESTIONS = [
    "Why is the decode phase memory-bandwidth-bound?",
    "What is the difference between prefill and decode in LLM inference?",
    "How does the KV cache reduce the cost of attention during decode?",
    "What problem does continuous batching solve compared to static batching?",
    "How does PagedAttention reduce KV cache memory fragmentation?",
    "Why is prefix caching especially useful for RAG workloads?",
    "How does speculative decoding speed up generation without changing the output?",
    "What is the difference between TensorRT-LLM and ONNX Runtime?",
    "Why does INT4 AWQ quantization speed up decode?",
    "How does AWQ decide which weight channels to protect?",
    "What is the difference between weight-only and activation quantization?",
    "How does normalizing embeddings make inner product equal cosine similarity?",
    "What is the difference between symmetric and asymmetric retrieval?",
    "When should I use an HNSW index instead of a flat index in FAISS?",
    "Why does chunk overlap matter when splitting documents?",
    "What does fsync guarantee and why is it slow?",
    "How does write-ahead logging make database commits crash-safe?",
    "What is XDP and why is it fast for packet filtering?",
    "How does Rust achieve memory safety without a garbage collector?",
    "What is the difference between monomorphization and trait object dispatch?",
]

def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def pct(xs, p):
    return statistics.quantiles(xs, n=100)[p - 1] if len(xs) > 1 else xs[0]

def main():
    embedder = Embedder()
    index = VectorIndex.load(embedder.dim)
    gen = Generator()

    t_embed, t_search, t_prefill, t_decode, t_total = [], [], [], [], []
    decode_toks = []

    # warmup
    _ = gen.generate(gen.build_prompt("warmup", ["warmup context"]))
    cuda_sync()

    for q in QUESTIONS:
        cuda_sync(); s = time.perf_counter()
        qvec = embedder.embed_query(q)
        cuda_sync(); t_embed.append(time.perf_counter() - s)

        s = time.perf_counter()
        hits = index.search(qvec, CFG.top_k)
        t_search.append(time.perf_counter() - s)

        prompt = gen.build_prompt(q, [c for c, _ in hits])
        inputs = gen.tok(prompt, return_tensors="pt").to(gen.device)

        # prefill ≈ time to first token (generate just one new token)
        cuda_sync(); s = time.perf_counter()
        with torch.inference_mode():
            gen.model.generate(**inputs, max_new_tokens=1, do_sample=False,
                               pad_token_id=gen.tok.eos_token_id)
        cuda_sync(); prefill = time.perf_counter() - s
        t_prefill.append(prefill)

        # full generation
        cuda_sync(); s = time.perf_counter()
        with torch.inference_mode():
            out = gen.model.generate(**inputs, max_new_tokens=CFG.max_new_tokens,
                                     do_sample=False, pad_token_id=gen.tok.eos_token_id)
        cuda_sync(); full = time.perf_counter() - s

        n_new = out.shape[1] - inputs["input_ids"].shape[1]
        t_decode.append(full - prefill)
        decode_toks.append((n_new - 1) / max(full - prefill, 1e-6))
        t_total.append(t_embed[-1] + t_search[-1] + full)

    def report(name, xs):
        print(f"{name:9s} p50={pct(xs,50)*1000:8.1f}ms  "
              f"p95={pct(xs,95)*1000:8.1f}ms  mean={statistics.mean(xs)*1000:8.1f}ms")

    print(f"\n--- per-stage latency over {len(QUESTIONS)} queries ---")
    for n, xs in [("embed",t_embed),("search",t_search),
                  ("prefill",t_prefill),("decode",t_decode),("TOTAL",t_total)]:
        report(n, xs)

    total = statistics.mean(t_total)
    print("\n--- share of total (mean) ---")
    for n, xs in [("embed",t_embed),("search",t_search),
                  ("prefill",t_prefill),("decode",t_decode)]:
        print(f"{n:9s} {statistics.mean(xs)/total*100:5.1f}%")
    print(f"\ndecode throughput: {statistics.mean(decode_toks):.1f} tokens/s")

if __name__ == "__main__":
    main()

Overwriting bench.py


In [14]:
!python bench.py

2026-06-28 04:19:54.746183: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-28 04:19:54.819129: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

--- per-stage latency over 20 queries ---
embed     p50=    11.8ms  p95=    32.4ms  mean=    12.9ms
search    p50=     0.1ms  p95=     0.2ms  mean=     0.1ms
prefill   p50

In [25]:
%%writefile generator_vllm.py
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from config import CFG

class VLLMGenerator:
    def __init__(self, model_name: str = CFG.gen_model, quant: str | None = None):
        # quant: None=FP16/BF16, "fp8" (H100 only), "awq"/"gptq" (needs a prequantized checkpoint)
        self.tok = AutoTokenizer.from_pretrained(model_name)
        kwargs = dict(
            model=model_name,
            dtype="auto",                 # bf16 on H100/A100 (no more forced float16)
            gpu_memory_utilization=0.85,
            max_model_len=4096,
            enable_prefix_caching=True,   # RAG win: caches repeated system-prompt/context prefixes
        )
        if quant:
            kwargs["quantization"] = quant   # "fp8" calibrates on load; H100 has the hardware
        self.llm = LLM(**kwargs)

    def build_prompt(self, question: str, contexts: list[str]) -> str:
        context_block = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(contexts))
        system = (
            "You are a helpful assistant. Answer the question using ONLY the context "
            "below. If the answer isn't in the context, say you don't know. "
            "Cite sources like [1], [2] where relevant."
        )
        messages = [
            {"role": "system", "content": system},
            {"role": "user",
             "content": f"Context:\n{context_block}\n\nQuestion: {question}"},
        ]
        return self.tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)

    def generate(self, prompt: str, max_tokens: int = CFG.max_new_tokens,
                 fixed_len: bool = False) -> str:
        sp = SamplingParams(temperature=0.0, max_tokens=max_tokens, ignore_eos=fixed_len)
        return self.llm.generate([prompt], sp, use_tqdm=False)[0].outputs[0].text

    def generate_batch(self, prompts, max_tokens: int = CFG.max_new_tokens,
                       fixed_len: bool = True):
        sp = SamplingParams(temperature=0.0, max_tokens=max_tokens, ignore_eos=fixed_len)
        return self.llm.generate(prompts, sp, use_tqdm=False)

Overwriting generator_vllm.py


In [26]:
%%writefile bench_vllm.py
import time, statistics, sys
from embedder import Embedder
from index import VectorIndex
from generator_vllm import VLLMGenerator
from config import CFG

QUESTIONS = [
    "Why is the decode phase memory-bandwidth-bound?",
    "What is the difference between prefill and decode in LLM inference?",
    "How does the KV cache reduce the cost of attention during decode?",
    "What problem does continuous batching solve compared to static batching?",
    "How does PagedAttention reduce KV cache memory fragmentation?",
    "Why is prefix caching especially useful for RAG workloads?",
    "How does speculative decoding speed up generation without changing the output?",
    "What is the difference between TensorRT-LLM and ONNX Runtime?",
    "Why does INT4 AWQ quantization speed up decode?",
    "How does AWQ decide which weight channels to protect?",
    "What is the difference between weight-only and activation quantization?",
    "How does normalizing embeddings make inner product equal cosine similarity?",
    "What is the difference between symmetric and asymmetric retrieval?",
    "When should I use an HNSW index instead of a flat index in FAISS?",
    "Why does chunk overlap matter when splitting documents?",
    "What does fsync guarantee and why is it slow?",
    "How does write-ahead logging make database commits crash-safe?",
    "What is XDP and why is it fast for packet filtering?",
    "How does Rust achieve memory safety without a garbage collector?",
    "What is the difference between monomorphization and trait object dispatch?",
]
MAX_TOKENS = 256

def pct(xs, p):
    return statistics.quantiles(xs, n=100)[p - 1] if len(xs) > 1 else xs[0]

def main():
    quant = sys.argv[1] if len(sys.argv) > 1 else None   # None | fp8
    embedder = Embedder()
    index = VectorIndex.load(embedder.dim)
    gen = VLLMGenerator(quant=quant)

    # real retrieval -> realistic prompt lengths
    prompts = []
    for q in QUESTIONS:
        hits = index.search(embedder.embed_query(q), CFG.top_k)
        prompts.append(gen.build_prompt(q, [c for c, _ in hits]))

    gen.generate(prompts[0], fixed_len=True)   # warmup, discard

    # --- single stream: fixed length kills the answer-length confound ---
    lat, tps = [], []
    for p in prompts:
        s = time.perf_counter()
        out = gen.generate_batch([p], MAX_TOKENS, fixed_len=True)
        dt = time.perf_counter() - s
        n = len(out[0].outputs[0].token_ids)
        lat.append(dt); tps.append(n / dt)

    print(f"\n=== vLLM  quant={quant} ===")
    print(f"single-stream: {statistics.mean(tps):6.1f} tokens/s   "
          f"p50={pct(lat,50)*1000:.0f}ms  p95={pct(lat,95)*1000:.0f}ms")

    # --- concurrency sweep: continuous batching amortizing weight reads ---
    print("--- concurrency sweep ---")
    for n in [1, 2, 4, 8, 16, 32, 64]:
        batch = (prompts * (n // len(prompts) + 1))[:n]
        s = time.perf_counter()
        outs = gen.generate_batch(batch, MAX_TOKENS, fixed_len=True)
        dt = time.perf_counter() - s
        toks = sum(len(o.outputs[0].token_ids) for o in outs)
        print(f"N={n:2d}  {toks/dt:8.1f} tokens/s")

if __name__ == "__main__":
    main()

Overwriting bench_vllm.py


In [27]:
!python bench_vllm.py

2026-06-28 04:34:03.042631: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-28 04:34:03.112510: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 06-28 04:34:07 [__init__.py:216] Automatically detected platform cuda.
INFO 06-28 04:34:14 [utils.py:233] non-default args: {'max_model_len': 4096, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
INFO 06-28 04:34:15 [model.py:547] Resolved architecture: Qwen2ForCausalLM
`torch_dtype` is 

In [28]:
!python bench_vllm.py bitsandbytes

2026-06-28 04:44:38.983951: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-28 04:44:39.053206: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 06-28 04:44:43 [__init__.py:216] Automatically detected platform cuda.
INFO 06-28 04:44:49 [utils.py:233] non-default args: {'max_model_len': 4096, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
INFO 06-28 04:44:51 [model.py:547] Resolved architecture: Qw

In [29]:
!python bench_vllm.py fp8

2026-06-28 04:46:39.126910: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-28 04:46:39.200904: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 06-28 04:46:43 [__init__.py:216] Automatically detected platform cuda.
INFO 06-28 04:46:50 [utils.py:233] non-default args: {'max_model_len': 4096, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'quantization': 'fp8', 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
INFO 06-28 04:46:51 [model.py:547] Resolved architecture: Qwen2ForCau

In [30]:
!/opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/nsys --version

NVIDIA Nsight Systems version 2025.1.1.0


In [35]:
%%writefile profile_decode.py
from vllm import LLM, SamplingParams
from config import CFG

def main():
    llm = LLM(model=CFG.gen_model, quantization="bitsandbytes", dtype="auto",
              gpu_memory_utilization=0.85, max_model_len=4096,
              enforce_eager=False)
    sp = SamplingParams(temperature=0.0, max_tokens=256, ignore_eos=True)

    prompt = "Hello"                         # tiny prompt = decode-dominated, minimal prefill
    llm.generate([prompt], sp)               # profiled region

if __name__ == "__main__":
    main()

Overwriting profile_decode.py


In [36]:
NSYS = "/opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/nsys"
!{NSYS} profile -o decode --force-overwrite true --trace=cuda python profile_decode.py
!{NSYS} stats --force-export=true --report cuda_gpu_kern_sum decode.nsys-rep

Invalid plugin configuration: Executable path does not exist: /opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/plugins/efa_metrics/nic_sampler
INFO 06-28 05:02:45 [__init__.py:216] Automatically detected platform cuda.
2026-06-28 05:02:46.810785: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-28 05:02:46.880098: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 06-28 05:02:53 [utils.py:233] non-default args: {'max_model_len': 4096, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'quantizat

In [37]:
%%writefile judge_quant.py
import os, json, random, statistics
from anthropic import Anthropic
from embedder import Embedder
from index import VectorIndex
from generator_vllm import VLLMGenerator
from config import CFG

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
JUDGE_MODEL = "claude-sonnet-4-6"

QUESTIONS = [
    "Why is the decode phase memory-bandwidth-bound?",
    "What is the difference between prefill and decode in LLM inference?",
    "How does the KV cache reduce the cost of attention during decode?",
    "What problem does continuous batching solve compared to static batching?",
    "How does PagedAttention reduce KV cache memory fragmentation?",
    "Why is prefix caching especially useful for RAG workloads?",
    "What is the difference between TensorRT-LLM and ONNX Runtime?",
    "Why does INT4 AWQ quantization speed up decode?",
    "What is the difference between weight-only and activation quantization?",
    "When should I use an HNSW index instead of a flat index in FAISS?",
    "What does fsync guarantee and why is it slow?",
    "How does write-ahead logging make database commits crash-safe?",
    "What is XDP and why is it fast for packet filtering?",
    "How does Rust achieve memory safety without a garbage collector?",
    "What is the difference between monomorphization and trait object dispatch?",
]

JUDGE_PROMPT = """You are evaluating two AI answers to a question, using ONLY the provided context.

Question:
{question}

Context the answers were given:
{context}

Answer A:
{answer_a}

Answer B:
{answer_b}

Judge which answer is better, considering: factual correctness given the context,
faithfulness (no claims unsupported by the context), and how well it answers the question.

Respond with ONLY a JSON object, no other text:
{{"winner": "A" | "B" | "tie", "reason": "<one short sentence>"}}"""

def judge_pair(question, context, ans_a, ans_b):
    msg = client.messages.create(
        model=JUDGE_MODEL, max_tokens=200,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(
            question=question, context=context, answer_a=ans_a, answer_b=ans_b)}],
    )
    text = msg.content[0].text.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    return json.loads(text)

def main():
    embedder = Embedder()
    index = VectorIndex.load(embedder.dim)

    # 1) cache retrieval ONCE so both generators see identical context
    contexts, prompts = [], []
    for q in QUESTIONS:
        hits = index.search(embedder.embed_query(q), CFG.top_k)
        chunks = [c for c, _ in hits]
        contexts.append("\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(chunks)))

    # 2) generate FP16 answers, free the engine, then FP8 answers
    #    (one engine at a time to avoid double GPU memory)
    print("generating FP16 (unquantized)...")
    gen = VLLMGenerator(quant=None)
    fp16 = [gen.generate(gen.build_prompt(q, _split(ctx)), max_tokens=256)
            for q, ctx in zip(QUESTIONS, contexts)]
    del gen
    import gc, torch; gc.collect(); torch.cuda.empty_cache()

    print("generating FP8 (quantized)...")
    gen = VLLMGenerator(quant="fp8")
    fp8 = [gen.generate(gen.build_prompt(q, _split(ctx)), max_tokens=256)
           for q, ctx in zip(QUESTIONS, contexts)]
    del gen

    # 3) blinded, position-randomized pairwise judging
    print("\njudging...\n")
    fp16_wins = fp8_wins = ties = 0
    for q, ctx, a16, a8 in zip(QUESTIONS, contexts, fp16, fp8):
        flip = random.random() < 0.5          # randomize which is shown as A
        a, b = (a16, a8) if not flip else (a8, a16)
        v = judge_pair(q, ctx, a, b)
        w = v["winner"]
        # map A/B back to which model it actually was
        if w == "tie":
            ties += 1; tag = "tie"
        else:
            chose_a = (w == "A")
            chose_fp16 = (chose_a != flip)    # untangle the shuffle
            if chose_fp16: fp16_wins += 1; tag = "FP16"
            else:          fp8_wins  += 1; tag = "FP8"
        print(f"[{tag:4}] {q[:55]}")

    n = len(QUESTIONS)
    print(f"\n=== quality: FP16 vs FP8 over {n} questions ===")
    print(f"FP16 (unquantized) wins: {fp16_wins}")
    print(f"FP8  (quantized)   wins: {fp8_wins}")
    print(f"ties:                    {ties}")
    print(f"\nverdict: {'quality preserved (mostly ties / balanced)' if abs(fp16_wins-fp8_wins) <= max(1, n//10) else 'meaningful quality difference — investigate'}")

def _split(ctx_block):
    # turn the cached "[1] ...\n\n[2] ..." block back into a chunk list for build_prompt
    import re
    parts = re.split(r"\n\n\[\d+\]\s*", ctx_block)
    parts[0] = re.sub(r"^\[\d+\]\s*", "", parts[0])
    return parts

if __name__ == "__main__":
    main()

Overwriting judge_quant.py


In [34]:
import os
os.environ["ANTHROPIC_API_KEY"] = ""   #key
!ANTHROPIC_API_KEY=$ANTHROPIC_API_KEY python judge_quant.py

2026-06-28 04:52:06.754817: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-28 04:52:06.825526: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 06-28 04:52:11 [__init__.py:216] Automatically detected platform cuda.
generating FP16 (unquantized)...
INFO 06-28 04:52:18 [utils.py:233] non-default args: {'max_model_len': 4096, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
INFO 06-28 04:52:19 [model.py:547] Resolved architecture: Q